## Local Windows setup

Select the **Deep RL Unit 3 (alias for main)** kernel. Unit 5 uses Docker Desktop and WSL 2 to run the course Linux builds; start Docker Desktop before running the notebook. Training and publishing start only when you run their cells.


# Unit 5: An Introduction to ML-Agents



<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit7/thumbnail.png" alt="Thumbnail"/>

In this notebook, you'll learn about ML-Agents and train two agents.

- The first one will learn to **shoot snowballs onto spawning targets**.
- The second need to press a button to spawn a pyramid, then navigate to the pyramid, knock it over, **and move to the gold brick at the top**. To do that, it will need to explore its environment, and we will use a technique called curiosity.

After that, you'll be able **to watch your agents playing directly on your browser**.

For more information about the certification process, check this section 👉 https://huggingface.co/deep-rl-course/en/unit0/introduction#certification-process

⬇️ Here is an example of what **you will achieve at the end of this unit.** ⬇️


<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit7/pyramids.gif" alt="Pyramids"/>

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit7/snowballtarget.gif" alt="SnowballTarget"/>

### 🎮 Environments:

- [Pyramids](https://github.com/Unity-Technologies/ml-agents/blob/main/docs/Learning-Environment-Examples.md#pyramids)
- SnowballTarget

### 📚 RL-Library:

- [ML-Agents](https://github.com/Unity-Technologies/ml-agents)


We're constantly trying to improve our tutorials, so **if you find some issues in this notebook**, please [open an issue on the GitHub Repo](https://github.com/huggingface/deep-rl-class/issues).

## Objectives of this notebook 🏆

At the end of the notebook, you will:

- Understand how works **ML-Agents**, the environment library.
- Be able to **train agents in Unity Environments**.


## This notebook is from the Deep Reinforcement Learning Course
<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/deep-rl-course-illustration.jpg" alt="Deep RL Course illustration"/>

In this free course, you will:

- 📖 Study Deep Reinforcement Learning in **theory and practice**.
- 🧑‍💻 Learn to **use famous Deep RL libraries** such as Stable Baselines3, RL Baselines3 Zoo, CleanRL and Sample Factory 2.0.
- 🤖 Train **agents in unique environments**

And more check 📚 the syllabus 👉 https://huggingface.co/deep-rl-course/communication/publishing-schedule

Don’t forget to **<a href="http://eepurl.com/ic5ZUD">sign up to the course</a>** (we are collecting your email to be able to **send you the links when each Unit is published and give you information about the challenges and updates).**


The best way to keep in touch is to join our discord server to exchange with the community and with us 👉🏻 https://discord.gg/ydHrjt3WP5

## Prerequisites 🏗️
Before diving into the notebook, you need to:

🔲 📚 **Study [what is ML-Agents and how it works by reading Unit 5](https://huggingface.co/deep-rl-course/unit5/introduction)**  🤗  

# Let's train our agents 🚀

**To validate this hands-on for the certification process, you just need to push your trained models to the Hub**. There’s no results to attain to validate this one. But if you want to get nice results you can try to attain:

- For `Pyramids` : Mean Reward = 1.75
- For `SnowballTarget` : Mean Reward = 15 or 30 targets hit in an episode.


## Set the GPU 💪
- To **accelerate the agent's training, we'll use a GPU**. To do that, go to `Runtime > Change Runtime type`

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/gpu-step1.jpg" alt="GPU Step 1">

- `Hardware Accelerator > GPU`

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/gpu-step2.jpg" alt="GPU Step 2">

## Local setup

The course supplies Linux Unity executables. On Windows the Jupyter notebook runs in `venv-main`, while ML-Agents runs in a separate Linux container.


In [ ]:
import sys
import subprocess
from pathlib import Path
from zipfile import ZipFile
import requests

cwd = Path.cwd().resolve()
unit5_dir = cwd if (cwd / "Dockerfile").is_file() else cwd / "notebooks" / "unit5"
if not (unit5_dir / "Dockerfile").is_file():
    raise FileNotFoundError("Open from the repository root or notebooks/unit5.")
print("Notebook Python:", sys.executable)
print("Unit 5 directory:", unit5_dir)
subprocess.run(["docker", "info", "--format", "{{.ServerVersion}}"], check=True)


## ML-Agents environment

The container has its own Python 3.10.12 and ML-Agents 1.1.0. Building it downloads CUDA PyTorch and may take time. The shared Windows environment is unchanged.


In [ ]:
image = "deep-rl-unit5:1.1.0"
subprocess.run(["docker", "build", "-t", image, str(unit5_dir)], check=True)


Conda installation and Linux virtual-display commands are unnecessary on the Windows host.


In [ ]:
subprocess.run([
    "docker", "run", "--rm", "--gpus", "all", image, "python", "-c",
    "import sys, torch; print(sys.version); print('PyTorch:', torch.__version__); "
    "print('CUDA:', torch.cuda.is_available()); "
    "print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'not available')"
], check=True)


## Prepare the Unity environments

Archives and extracted environments are stored in the ignored directory `training-envs-executables/linux`. Training output is stored in `results/`.


In [ ]:
envs_dir = unit5_dir / "training-envs-executables" / "linux"
envs_dir.mkdir(parents=True, exist_ok=True)

def prepare_environment(name, url):
    archive = envs_dir / f"{name}.zip"
    root = envs_dir.resolve()
    if not archive.is_file():
        with requests.get(url, stream=True, timeout=120) as response:
            response.raise_for_status()
            with archive.open("wb") as output:
                for chunk in response.iter_content(1024 * 1024):
                    output.write(chunk)
    with ZipFile(archive) as bundle:
        for item in bundle.infolist():
            if not (root / item.filename).resolve().is_relative_to(root):
                raise ValueError(f"Unsafe archive member: {item.filename}")
        bundle.extractall(root)
    print("Ready:", name)


## SnowballTarget ⛄

If you need a refresher on how this environments work check this section 👉
https://huggingface.co/deep-rl-course/unit5/snowball-target

### Prepare SnowballTarget

The Linux build runs inside Docker.


The training configuration is `config/SnowballTarget.yaml`.


The following cell downloads the course archive if necessary.


In [ ]:
prepare_environment("SnowballTarget", "https://github.com/huggingface/Snowball-Target/raw/main/SnowballTarget.zip")


We unzip the executable.zip file

The archive is extracted by `prepare_environment`.


Make sure your file is accessible

Execution permissions are set inside the container after copying the build to its Linux filesystem.


### SnowballTarget configuration

The course example is in `config/SnowballTarget.yaml`. Modify the hyperparameters before training if desired.


<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit7/snowballfight_config1.png" alt="Config SnowballTarget"/>
<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit7/snowballfight_config2.png" alt="Config SnowballTarget"/>

As an experimentation, you should also try to modify some other hyperparameters. Unity provides very [good documentation explaining each of them here](https://github.com/Unity-Technologies/ml-agents/blob/main/docs/Training-Configuration-File.md).

Now that you've created the config file and understand what most hyperparameters do, we're ready to train our agent 🔥.

### Train SnowballTarget

The container copies the Unity build to `/tmp`; checkpoints and models are saved in `notebooks/unit5/results/`. Training uses the GPU.


The training will take 10 to 35min depending on your config, go take a ☕️you deserve it 🤗.

In [ ]:
def train_unity(run_id, config, game, executable):
    command = (
        f"cp -a /work/training-envs-executables/linux/{game} /tmp/{game} && "
        f"chmod +x /tmp/{game}/{executable} && "
        f"exec mlagents-learn /work/config/{config} --env=/tmp/{game}/{executable} "
        f"--run-id={run_id} --no-graphics"
    )
    subprocess.run([
        "docker", "run", "--rm", "--gpus", "all",
        "--mount", f"type=bind,source={unit5_dir},target=/work",
        "--workdir", "/work", image, "bash", "-lc", command,
    ], check=True)

train_unity("SnowballTarget1", "SnowballTarget.yaml", "SnowballTarget", "SnowballTarget.x86_64")


### Push the agent to the 🤗 Hub

- Now that we trained our agent, we’re **ready to push it to the Hub to be able to visualize it playing on your browser🔥.**

To be able to share your model with the community there are three more steps to follow:

1️⃣ (If it's not already done) create an account to HF ➡ https://huggingface.co/join

2️⃣ Sign in and then, you need to store your authentication token from the Hugging Face website.
- Create a new token (https://huggingface.co/settings/tokens) **with write role**

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/create-token.jpg" alt="Create HF Token">

- Copy the token
- Run the cell below and paste the token

In [ ]:
from huggingface_hub import get_token, notebook_login
if not get_token():
    notebook_login()
else:
    print("Hugging Face token found.")


Log in when you are ready to publish. A token with write access is required.


Then, we simply need to run `mlagents-push-to-hf`.

And we define 4 parameters:

1. `--run-id`: the name of the training run id.
2. `--local-dir`: where the agent was saved, it’s results/<run_id name>, so in my case results/First Training.
3. `--repo-id`: the name of the Hugging Face repo you want to create or update. It’s always <your huggingface username>/<the repo name>
If the repo does not exist **it will be created automatically**
4. `--commit-message`: since HF repos are git repository you need to define a commit message.

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit7/mlagentspushtohub.png" alt="Push to Hub"/>

For instance:

`!mlagents-push-to-hf  --run-id="SnowballTarget1" --local-dir="./results/SnowballTarget1" --repo-id="ThomasSimonini/ppo-SnowballTarget"  --commit-message="First Push"`

In [ ]:
import os
from huggingface_hub import get_token

def publish_run(run_id, repo_id):
    result_dir = unit5_dir / "results" / run_id
    if not result_dir.is_dir():
        raise FileNotFoundError(f"Training results not found: {result_dir}")
    token = get_token()
    if not token:
        raise RuntimeError("Log in to Hugging Face before publishing.")
    child_env = os.environ.copy()
    child_env["HF_TOKEN"] = token
    subprocess.run([
        "docker", "run", "--rm", "--mount", f"type=bind,source={unit5_dir},target=/work",
        "--workdir", "/work", "-e", "HF_TOKEN", image, "mlagents-push-to-hf",
        f"--run-id={run_id}", f"--local-dir=/work/results/{run_id}",
        f"--repo-id={repo_id}", "--commit-message=Upload trained agent",
    ], env=child_env, check=True)

publish_run("SnowballTarget1", "oncominglane/ppo-SnowballTarget")


Change `repo_id` above to publish to another repository.


Else, if everything worked you should have this at the end of the process(but with a different url 😆) :



```
Your model is pushed to the hub. You can view your model here: https://huggingface.co/ThomasSimonini/ppo-SnowballTarget
```

It’s the link to your model, it contains a model card that explains how to use it, your Tensorboard and your config file. **What’s awesome is that it’s a git repository, that means you can have different commits, update your repository with a new push etc.**

But now comes the best: **being able to visualize your agent online 👀.**

### Watch your agent playing 👀

For this step it’s simple:

1. Go here: https://huggingface.co/spaces/ThomasSimonini/ML-Agents-SnowballTarget

2. Launch the game and put it in full screen by clicking on the bottom right button

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit7/snowballtarget_load.png" alt="Snowballtarget load"/>

1. In step 1, type your username (your username is case sensitive: for instance, my username is ThomasSimonini not thomassimonini or ThOmasImoNInI) and click on the search button.

2. In step 2, select your model repository.

3. In step 3, **choose which model you want to replay**:
  - I have multiple ones, since we saved a model every 500000 timesteps.
  - But since I want the more recent, I choose `SnowballTarget.onnx`

👉 What’s nice **is to try with different models step to see the improvement of the agent.**

And don't hesitate to share the best score your agent gets on discord in #rl-i-made-this channel 🔥

Let's now try a harder environment called Pyramids...

## Pyramids

Prepare the second Linux Unity build for the same container.


This is the Pyramids archive used in the original course.


In [ ]:
prepare_environment("Pyramids", "https://huggingface.co/spaces/unity/ML-Agents-Pyramids/resolve/main/Pyramids.zip")


We unzip the executable.zip file

The archive is extracted by `prepare_environment`.


Make sure your file is accessible

Execution permissions are set inside the container.


### Pyramids configuration

`config/PyramidsRND.yaml` contains the course configuration with curiosity and `max_steps: 1000000`.


As an experimentation, you should also try to modify some other hyperparameters, Unity provides a very [good documentation explaining each of them here](https://github.com/Unity-Technologies/ml-agents/blob/main/docs/Training-Configuration-File.md).

We’re now ready to train our agent 🔥.

### Train the agent

The training will take 30 to 45min depending on your machine, go take a ☕️you deserve it 🤗.

In [ ]:
train_unity("PyramidsTraining", "PyramidsRND.yaml", "Pyramids", "Pyramids")


### Push the agent to the 🤗 Hub

- Now that we trained our agent, we’re **ready to push it to the Hub to be able to visualize it playing on your browser🔥.**

In [ ]:
publish_run("PyramidsTraining", "oncominglane/ppo-PyramidsRND")


### Watch your agent playing 👀

👉 https://huggingface.co/spaces/unity/ML-Agents-Pyramids

### 🎁 Bonus: Why not train on another environment?
Now that you know how to train an agent using MLAgents, **why not try another environment?**

MLAgents provides 17 different and we’re building some custom ones. The best way to learn is to try things of your own, have fun.



![cover](https://miro.medium.com/max/1400/0*xERdThTRRM2k_U9f.png)

You have the full list of the Unity official environments here 👉 https://github.com/Unity-Technologies/ml-agents/blob/develop/docs/Learning-Environment-Examples.md

For the demos to visualize your agent 👉 https://huggingface.co/unity

For now we have integrated:
- [Worm](https://huggingface.co/spaces/unity/ML-Agents-Worm) demo where you teach a **worm to crawl**.
- [Walker](https://huggingface.co/spaces/unity/ML-Agents-Walker) demo where you teach an agent **to walk towards a goal**.

That’s all for today. Congrats on finishing this tutorial!

The best way to learn is to practice and try stuff. Why not try another environment? ML-Agents has 17 different environments, but you can also create your own? Check the documentation and have fun!

See you on Unit 6 🔥,

## Keep Learning, Stay  awesome 🤗